# Baseball Batter's statistic prediction

In this notebook we will try to predict a baseball hitters statistic.

In [1]:
import pandas as pd

df_data = pd.read_csv('data/Batting.csv')

df_data.info()
df_data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101332 entries, 0 to 101331
Data columns (total 22 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   playerID  101332 non-null  object 
 1   yearID    101332 non-null  int64  
 2   stint     101332 non-null  int64  
 3   teamID    101332 non-null  object 
 4   lgID      100595 non-null  object 
 5   G         101332 non-null  int64  
 6   AB        96183 non-null   float64
 7   R         96183 non-null   float64
 8   H         96183 non-null   float64
 9   2B        96183 non-null   float64
 10  3B        96183 non-null   float64
 11  HR        96183 non-null   float64
 12  RBI       95759 non-null   float64
 13  SB        94883 non-null   float64
 14  CS        72729 non-null   float64
 15  BB        96183 non-null   float64
 16  SO        88345 non-null   float64
 17  IBB       59620 non-null   float64
 18  HBP       93373 non-null   float64
 19  SH        89845 non-null   float64
 20  SF  

,playerID,yearID,stint,teamID,lgID,G,AB,R,H,2B,...,RBI,SB,CS,BB,SO,IBB,HBP,SH,SF,GIDP
0,abercda01,1871,1,TRO,NaN,1,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
1,addybo01,1871,1,RC1,NaN,25,118.0,30.0,32.0,6.0,...,13.0,8.0,1.0,4.0,0.0,NaN,NaN,NaN,NaN,NaN
2,allisar01,1871,1,CL1,NaN,29,137.0,28.0,40.0,4.0,...,19.0,3.0,1.0,2.0,5.0,NaN,NaN,NaN,NaN,NaN
3,allisdo01,1871,1,WS3,NaN,27,133.0,28.0,44.0,10.0,...,27.0,1.0,1.0,0.0,2.0,NaN,NaN,NaN,NaN,NaN
4,ansonca01,1871,1,RC1,NaN,25,120.0,29.0,39.0,11.0,...,16.0,6.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,NaN


## Calculate Signles


In [2]:
df_data['1B'] = df_data['H'] - df_data['2B'] - df_data['3B'] - df_data['HR']
df_data['1B'] = df_data['1B'].fillna(0)
df_data['1B'] = df_data['1B'].astype(int)

df_data.head()

,playerID,yearID,stint,teamID,lgID,G,AB,R,H,2B,...,SB,CS,BB,SO,IBB,HBP,SH,SF,GIDP,1B
0,abercda01,1871,1,TRO,NaN,1,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0
1,addybo01,1871,1,RC1,NaN,25,118.0,30.0,32.0,6.0,...,8.0,1.0,4.0,0.0,NaN,NaN,NaN,NaN,NaN,26
2,allisar01,1871,1,CL1,NaN,29,137.0,28.0,40.0,4.0,...,3.0,1.0,2.0,5.0,NaN,NaN,NaN,NaN,NaN,31
3,allisdo01,1871,1,WS3,NaN,27,133.0,28.0,44.0,10.0,...,1.0,1.0,0.0,2.0,NaN,NaN,NaN,NaN,NaN,30
4,ansonca01,1871,1,RC1,NaN,25,120.0,29.0,39.0,11.0,...,6.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,NaN,25


# Cleaning

In [5]:
required_columns = ['BB', 'IBB', 'HBP', '1B', '2B', '3B', 'HR', 'AB', 'SF']
cleaned_df = df_data.dropna(subset=required_columns)
cleaned_df = cleaned_df[cleaned_df['AB'] >= 400]

cleaned_df.head()

,playerID,yearID,stint,teamID,lgID,G,AB,R,H,2B,...,SB,CS,BB,SO,IBB,HBP,SH,SF,GIDP,1B
37447,aaronha01,1955,1,ML1,NL,153,602.0,105.0,189.0,37.0,...,3.0,1.0,49.0,61.0,5.0,3.0,7.0,4.0,20.0,116
37465,ashburi01,1955,1,PHI,NL,140,533.0,91.0,180.0,32.0,...,12.0,10.0,105.0,36.0,5.0,3.0,2.0,1.0,3.0,136
37468,avilabo01,1955,1,CLE,AL,141,537.0,83.0,146.0,22.0,...,1.0,4.0,82.0,47.0,1.0,2.0,18.0,4.0,17.0,107
37472,bakerge02,1955,1,CHN,NL,154,609.0,82.0,163.0,29.0,...,9.0,7.0,49.0,57.0,1.0,2.0,18.0,3.0,14.0,116
37473,bankser01,1955,1,CHN,NL,154,596.0,98.0,176.0,29.0,...,9.0,3.0,45.0,72.0,6.0,2.0,0.0,3.0,16.0,94


# Feature Engineering

In [7]:
cleaned_df = cleaned_df.sort_values(by=['playerID', 'yearID'])

for col in required_columns:
    cleaned_df[f'{col}_lag1'] = cleaned_df.groupby('playerID')[col].shift(1)
    cleaned_df[f'{col}_lag2'] = cleaned_df.groupby('playerID')[col].shift(2)

cleaned_df = cleaned_df.dropna(subset=[f'{col}_lag1' for col in required_columns] + 
                                        [f'{col}_lag2' for col in required_columns])

cleaned_df.head()

,playerID,yearID,stint,teamID,lgID,G,AB,R,H,2B,...,2B_lag1,2B_lag2,3B_lag1,3B_lag2,HR_lag1,HR_lag2,AB_lag1,AB_lag2,SF_lag1,SF_lag2
39976,aaronha01,1959,1,ML1,NL,154,629.0,116.0,223.0,46.0,...,34.0,27.0,4.0,6.0,30.0,44.0,601.0,615.0,3.0,3.0
40608,aaronha01,1960,1,ML1,NL,153,590.0,102.0,172.0,20.0,...,46.0,34.0,7.0,4.0,39.0,30.0,629.0,601.0,9.0,3.0
41245,aaronha01,1961,1,ML1,NL,155,603.0,115.0,197.0,39.0,...,20.0,46.0,11.0,7.0,40.0,39.0,590.0,629.0,12.0,9.0
41943,aaronha01,1962,1,ML1,NL,156,592.0,127.0,191.0,28.0,...,39.0,20.0,10.0,11.0,34.0,40.0,603.0,590.0,9.0,12.0
42703,aaronha01,1963,1,ML1,NL,161,631.0,121.0,201.0,29.0,...,28.0,39.0,6.0,10.0,45.0,34.0,592.0,603.0,6.0,9.0


# Data splitting

In [8]:
split_year = 2010

train_df = cleaned_df[cleaned_df['yearID'] < split_year]
val_df = cleaned_df[cleaned_df['yearID'] >= split_year]

# Training

In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

feature_cols = ['HR_lag1', 'HR_lag2']
target_col = 'HR'

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_val = val_df[feature_cols]
y_val = val_df[target_col]

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

mse = mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R² Score: {r2:.3f}")

Mean Squared Error: 63.11
R² Score: 0.333
